#### **Milestone 3 setup (run first, before answering any questions)**
Run this code in a code cell
 before answering any of the question. This will create your knowledge
base and the FAISS index for this milestone.

In [100]:
!pip install -qq faiss-cpu # install FAISS

In [101]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base successfully created


#### Zero-shot classifier for Q1, Q2, Q6

In [102]:
# Zero-shot classifier for Q1, Q2, Q6

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

#### Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [103]:
result = zs(prompt_150, labels_150)

print(f"The predicted probability score assigned to the ground truth is: {result['scores'][result['labels'].index(ans_150)]:.3f}")

The predicted probability score assigned to the ground truth is: 0.384


#### Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [104]:
q2_embedding = model.encode([prompt_150]).astype('float32')

distances, retrieved_indices = index.search(q2_embedding, k=10)

target_idx = 150
rank = -1

if target_idx in retrieved_indices[0]:
    rank = np.where(retrieved_indices[0] == target_idx)[0][0] + 1

print(f"Indices of top 10 results: {retrieved_indices[0]}")
print(f"The exact rank of the true correct document (index 150) is: {rank}")

Indices of top 10 results: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
The exact rank of the true correct document (index 150) is: 10


#### Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [105]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices[0]] # Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] # Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [106]:
indexed_scores = list(zip(retrieved_ids, ce_scores))
ranked_results = sorted(indexed_scores, key=lambda x: x[1], reverse=True)

target_idx = 150

ce_rank = -1

for i, (idx, score) in enumerate(ranked_results):
    if idx == target_idx:
        ce_rank = i + 1
        break

print(f"Cross-Encoder Sorted Indices: {[x[0] for x in ranked_results]}")
print(f"The exact rank of the true correct document according to the Cross-Encoder is: {ce_rank}")

Cross-Encoder Sorted Indices: [np.int64(150), np.int64(847), np.int64(1693), np.int64(1906), np.int64(1269), np.int64(1532), np.int64(168), np.int64(576), np.int64(663), np.int64(1701)]
The exact rank of the true correct document according to the Cross-Encoder is: 1


#### Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [107]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

q4_embeddings = model.encode([prompt_42]).astype('float32')
q4_distances, q4_retrieved_indices = index.search(q4_embeddings, k=5)

concatenated_docs = ' '.join([kb[i] for i in q4_retrieved_indices[0]])

concatenated_docs_prompt = f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens = tokenizer.tokenize(concatenated_docs_prompt)

print(f"The total number of tokens generated is: {len(tokens)}")

The total number of tokens generated is: 214


#### Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [108]:
true_document = kb[150]
rag_string = f"Context: {true_document} Question: {prompt_150}"
result = zs(rag_string, labels_150)

print(f"The new predicted probability score of the ground-truth correct option is: {result['scores'][result['labels'].index(ans_150)]:.3f}")

The new predicted probability score of the ground-truth correct option is: 0.989


#### Concept: The Danger of Bad Retrieval (Adversarial RAG)
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

#### Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [109]:
adversarial_rag_string = f"Context: {kb[999]} Question: {prompt_150}"
result = zs(adversarial_rag_string, labels_150)

prob_score = result['scores'][result['labels'].index(ans_150)]
print(f"The new predicted probability score of the ground-truth correct option is: {prob_score:.3f}")

The new predicted probability score of the ground-truth correct option is: 0.529


#### In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [110]:
hits = 0
n_rows = 100

subset = train.iloc[:n_rows]
prompts = subset['prompt'].tolist()
true_answers = [str(row[row['answer']]) for _, row in subset.iterrows()]

prompt_embeddings = model.encode(prompts, show_progress_bar=False).astype('float32')

distances, retrieved_indices = index.search(prompt_embeddings, k=5)

for i in range(n_rows):
    retrieved_docs = [kb[idx] for idx in retrieved_indices[i]]

    if any(true_answers[i] in doc for doc in retrieved_docs):
        hits += 1

hit_rate = (hits / n_rows) * 100
print(f"The Hit Rate percentage for the first 100 rows is: {hit_rate:.1f}%")

The Hit Rate percentage for the first 100 rows is: 73.0%


#### Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.

For each row, your pipeline must do the following in order:

**Retrieve:** Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

**Rerank:** Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

**Augment:** Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

**Predict:** Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

**Score:** Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [111]:
def calculate_map3(true_answer, predicted_labels):
    score = 0.0
    for i, label in enumerate(predicted_labels[:3]):
        if label == true_answer:
            score = 1.0 / (i + 1)
            break
    return score

subset_20 = train.iloc[:20]
all_map_scores = []

for idx, row in subset_20.iterrows():
    p_emb = model.encode([row['prompt']], show_progress_bar=False).astype('float32')
    _, retrieved_indices = index.search(p_emb, k=5)
    docs_5 = [kb[i] for i in retrieved_indices[0]]

    pairs = [[row['prompt'], doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    best_doc_idx = retrieved_indices[0][np.argmax(ce_scores)]
    best_doc = kb[best_doc_idx]

    rag_str = f"Context: {best_doc} Question: {row['prompt']}"

    candidate_labels = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
    result = zs(rag_str, candidate_labels)

    label_to_letter = {str(row[opt]): opt for opt in ['A', 'B', 'C', 'D', 'E']}
    top_3_letters = [label_to_letter[label] for label in result['labels'][:3]]

    actual_answer = row['answer']
    row_score = calculate_map3(actual_answer, top_3_letters)
    all_map_scores.append(row_score)

print(f"The final average MAP@3 score is: {np.mean(all_map_scores):.3f}")

The final average MAP@3 score is: 0.975
